## Setting things up

### Import libraries
Add directory with the CIBUSmod modules to path to be able to import

In [1]:
import sys
import os
sys.path.insert(0, 'C:\\Users/jnka0003/Git repos/CIBUSmod')

Import CIBUSmod and packages for handling data and plotting

In [2]:
import CIBUSmod as cm

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

### Set up scenarios

In [3]:
# Create session
session = cm.Session(
    name = 'FORMAS_sens',
    data_path = 'C:\\Users/jnka0003/Git repos/CIBUSmod/data',
    data_path_scenarios = 'scenarios',
    data_path_output = 'output',
)

# Define scenarios
session.add_scenario(
    name = 'BL',
    scenario_workbooks = 'BASELINE', 
    modules = 'all',
    pars = 'all',
    years = 0
)
session.add_scenario(
    name = 'MAX_CUR + FIX_ANI',
    scenario_workbooks = ['COMMON'], 
    modules = 'all',
    pars = 'all',
    years = [100]
)
session.add_scenario(
    name = 'ALL + MORE_WOODED',
    scenario_workbooks = ['COMMON', 'STEERS', 'CULandDRY_COWS',
                          'WIN_LAMB', 'REC_HORSES', 'SENS_MORE_WOODED'],
    modules = 'all',
    pars = 'all',
    years = [70,100,110]
)
session.add_scenario(
    name = 'ALL + LOW_YIELDS',
    scenario_workbooks = ['COMMON', 'STEERS', 'CULandDRY_COWS',
                          'WIN_LAMB', 'REC_HORSES', 'SENS_LOW_YIELDS'],
    modules = 'all',
    pars = 'all',
    years = [70,100,110]
)
session.add_scenario(
    name = 'ALL + HIGH_SHARE_SNG',
    scenario_workbooks = ['COMMON', 'STEERS', 'CULandDRY_COWS',
                          'WIN_LAMB', 'REC_HORSES', 'SENS_HIGH_SHARE_SNG'],
    modules = 'all',
    pars = 'all',
    years = [70,100,110]
)
session.add_scenario(
    name = 'ALL + MORE_POT',
    scenario_workbooks = ['COMMON', 'STEERS', 'CULandDRY_COWS',
                          'WIN_LAMB', 'REC_HORSES', 'SENS_MORE_POT'],
    modules = 'all',
    pars = 'all',
    years = [70,100,110]
)
session.add_scenario(
    name = 'ALL + MORE_CROPLAND',
    scenario_workbooks = ['COMMON', 'STEERS', 'CULandDRY_COWS',
                          'WIN_LAMB', 'REC_HORSES', 'SENS_MORE_CROPLAND'],
    modules = 'all',
    pars = 'all',
    years = [70,100,110]
)


A scenario with the name 'BL' already exists use .update_scenario() or .remove_scenario() instead.
A scenario with the name 'MAX_CUR + FIX_ANI' already exists use .update_scenario() or .remove_scenario() instead.
A scenario with the name 'ALL + MORE_WOODED' already exists use .update_scenario() or .remove_scenario() instead.
A scenario with the name 'ALL + LOW_YIELDS' already exists use .update_scenario() or .remove_scenario() instead.
A scenario with the name 'ALL + HIGH_SHARE_SNG' already exists use .update_scenario() or .remove_scenario() instead.
A scenario with the name 'ALL + MORE_POT' already exists use .update_scenario() or .remove_scenario() instead.
A scenario with the name 'ALL + MORE_CROPLAND' already exists use .update_scenario() or .remove_scenario() instead.


## Run scenarios (multi proc.)

In [4]:
# Import
from concurrent.futures import ProcessPoolExecutor, as_completed
from multi_proc import do_run

In [5]:
# Create list of scenarios to run
# runs = [(s,y) for s,y in session.iterate('2025-05-14')]
runs = [(s,y) for s,y in session.iterate('all')]
# runs = [(s,y) for s,y in session.iterate('2025-05-02')]
# runs = [(s,y) for s,y in runs if 'MORE_WOODED' in s]
# runs = [(s,y) for s,y in runs if 'ALL' in s and 'FIX_ANI' in s]
runs

[('BL', '0'),
 ('MAX_CUR + FIX_ANI', '100'),
 ('ALL + MORE_WOODED', '70'),
 ('ALL + MORE_WOODED', '100'),
 ('ALL + MORE_WOODED', '110'),
 ('ALL + LOW_YIELDS', '70'),
 ('ALL + LOW_YIELDS', '100'),
 ('ALL + LOW_YIELDS', '110'),
 ('ALL + HIGH_SHARE_SNG', '70'),
 ('ALL + HIGH_SHARE_SNG', '100'),
 ('ALL + HIGH_SHARE_SNG', '110'),
 ('ALL + MORE_POT', '70'),
 ('ALL + MORE_POT', '100'),
 ('ALL + MORE_POT', '110'),
 ('ALL + MORE_CROPLAND', '70'),
 ('ALL + MORE_CROPLAND', '100'),
 ('ALL + MORE_CROPLAND', '110')]

In [6]:
%%time
# Do the multi-processing
with ProcessPoolExecutor(max_workers=8) as executor:
    
    futures = {executor.submit(do_run, session, scn_year) : scn_year for scn_year in runs}

    for future in as_completed(futures):
    
        scn, year = futures[future]
           
        try:
            t = future.result()
        except Exception as ee:
            print(f'(!!!) {scn}, {year} failed with the exception: {ee}')
        else:
            m = int(t/60)
            s = int(round(t - m*60))
            print(f'{scn}, {year} finished successfully in {m}min {s}s')
            
session.cache.clear()

BL, 0 finished successfully in 5min 27s
ALL + LOW_YIELDS, 110 finished successfully in 8min 12s
ALL + MORE_WOODED, 100 finished successfully in 8min 53s
ALL + LOW_YIELDS, 70 finished successfully in 9min 4s
ALL + MORE_WOODED, 110 finished successfully in 9min 7s
MAX_CUR + FIX_ANI, 100 finished successfully in 9min 17s
ALL + MORE_WOODED, 70 finished successfully in 9min 19s
ALL + LOW_YIELDS, 100 finished successfully in 9min 25s
ALL + HIGH_SHARE_SNG, 70 finished successfully in 4min 41s
ALL + HIGH_SHARE_SNG, 100 finished successfully in 5min 57s
ALL + MORE_POT, 100 finished successfully in 7min 48s
ALL + MORE_CROPLAND, 70 finished successfully in 7min 55s
ALL + HIGH_SHARE_SNG, 110 finished successfully in 9min 5s
ALL + MORE_POT, 110 finished successfully in 8min 41s
ALL + MORE_POT, 70 finished successfully in 9min 2s
ALL + MORE_CROPLAND, 100 finished successfully in 8min 38s
ALL + MORE_CROPLAND, 110 finished successfully in 7min 52s
CPU times: total: 46.9 ms
Wall time: 19min 13s
